**MGMT298D: Science and Strategy of AI**
# Week 6I: TinyStories Text Completion

This instructor notebook trains a small GPT-style decoder on TinyStories. The goal is not to build a large general-purpose LLM, but to train a compact model that can produce visibly coherent short text on a Colab T4.

# Setup

In [ ]:
#@title Install dependencies { display-mode: "form" }
!pip -q install datasets


In [ ]:
#@title Import libraries { display-mode: "form" }
import json
import os
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from datasets import load_dataset
from IPython.display import HTML, display
from tensorflow import keras
from tensorflow.keras import layers
import ipywidgets as widgets

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
tf.keras.mixed_precision.set_global_policy("mixed_float16")

In [ ]:
#@title Saved model options { display-mode: "form" }
LOAD_SAVED_WEIGHTS = False  #@param {type:"boolean"}
SAVE_DIR = "week6i_tinystories_model"
WEIGHTS_PATH = os.path.join(SAVE_DIR, "tinystories_tiny_gpt.weights.h5")
VOCAB_PATH = os.path.join(SAVE_DIR, "vocab.json")
CONFIG_PATH = os.path.join(SAVE_DIR, "config.json")
ZIP_PATH = f"{SAVE_DIR}.zip"

---
# Load TinyStories

TinyStories is a dataset of short, simple stories designed so very small language models can still learn coherent English.

In [ ]:
#@title Load corpus { display-mode: "form" }
TRAIN_STORIES = 100_000  #@param {type:"integer"}
VAL_STORIES = 5_000     #@param {type:"integer"}

train_raw = load_dataset("roneneldan/TinyStories", split=f"train[:{TRAIN_STORIES}]")
val_raw = load_dataset("roneneldan/TinyStories", split=f"validation[:{VAL_STORIES}]")

train_texts = train_raw["text"]
val_texts = val_raw["text"]

In [ ]:
#@title Peek at training examples { display-mode: "form" }
for i, story in enumerate(train_texts[:3], start=1):
    preview = story.replace("\n", " ").strip()
    if len(preview) > 700:
        preview = preview[:700].rstrip() + "..."
    print(f"Story {i}\n{preview}\n")


---
# Model Settings

In [ ]:
#@title Shared settings { display-mode: "form" }
VOCAB_SIZE = 8000
SEQ_LEN = 128
BATCH_SIZE = 64
EPOCHS = 2
DEFAULT_PROMPT = "Once upon a time"
GENERATED_TOKENS = 60
DEFAULT_TEMPERATURE = 0.8

EMBED_DIM = 256
NUM_HEADS = 8
FF_DIM = 1024
NUM_BLOCKS = 6
DROPOUT = 0.1
LEARNING_RATE = 3e-4

In [ ]:
#@title Architecture overview { display-mode: "form" }
print(
    f"Tiny GPT-style model: {NUM_BLOCKS} blocks, {NUM_HEADS} heads, "
    f"{EMBED_DIM} embedding dim, {SEQ_LEN}-token context, {VOCAB_SIZE:,}-word vocabulary."
)

---
# Data Pipeline

In [ ]:
#@title Build language-model datasets { display-mode: "form" }
def lowercase_keep_punctuation(text):
    return tf.strings.lower(text)

vectorizer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    standardize=lowercase_keep_punctuation,
    output_sequence_length=SEQ_LEN + 1,
)

if LOAD_SAVED_WEIGHTS and os.path.exists(VOCAB_PATH):
    with open(VOCAB_PATH, "r", encoding="utf-8") as f:
        vectorizer.set_vocabulary(json.load(f))
else:
    vectorizer.adapt(tf.data.Dataset.from_tensor_slices(train_texts).batch(1024))

vocab = vectorizer.get_vocabulary()


def make_lm_batch(text_batch):
    tokens = vectorizer(text_batch)
    return tokens[:, :-1], tokens[:, 1:]


autotune = tf.data.AUTOTUNE
train_ds = (
    tf.data.Dataset.from_tensor_slices(train_texts)
    .shuffle(20_000, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .map(make_lm_batch, num_parallel_calls=autotune)
    .prefetch(autotune)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices(val_texts)
    .batch(BATCH_SIZE)
    .map(make_lm_batch, num_parallel_calls=autotune)
    .prefetch(autotune)
)

---
# Build the Tiny GPT

In [ ]:
#@title Define model { display-mode: "form" }
def transformer_block(x, block_id):
    attn_input = layers.LayerNormalization(name=f"block_{block_id}_attn_norm")(x)
    attn_output = layers.MultiHeadAttention(
        num_heads=NUM_HEADS,
        key_dim=EMBED_DIM // NUM_HEADS,
        dropout=DROPOUT,
        name=f"block_{block_id}_causal_attention",
    )(attn_input, attn_input, use_causal_mask=True)
    x = x + layers.Dropout(DROPOUT, name=f"block_{block_id}_attn_dropout")(attn_output)

    ff_input = layers.LayerNormalization(name=f"block_{block_id}_ff_norm")(x)
    ff_output = layers.Dense(FF_DIM, activation="gelu", name=f"block_{block_id}_ff_dense_1")(ff_input)
    ff_output = layers.Dense(EMBED_DIM, name=f"block_{block_id}_ff_dense_2")(ff_output)
    x = x + layers.Dropout(DROPOUT, name=f"block_{block_id}_ff_dropout")(ff_output)
    return x


def build_model():
    tokens = layers.Input(shape=(SEQ_LEN,), dtype="int32", name="tokens")
    positions = tf.range(start=0, limit=SEQ_LEN, delta=1)

    token_embeddings = layers.Embedding(VOCAB_SIZE, EMBED_DIM, name="token_embedding")(tokens)
    position_embeddings = layers.Embedding(SEQ_LEN, EMBED_DIM, name="position_embedding")(positions)
    x = layers.Dropout(DROPOUT, name="embedding_dropout")(token_embeddings + position_embeddings)

    for block_id in range(1, NUM_BLOCKS + 1):
        x = transformer_block(x, block_id)

    x = layers.LayerNormalization(name="final_norm")(x)
    logits = layers.Dense(VOCAB_SIZE, dtype="float32", name="next_token_logits")(x)
    return keras.Model(tokens, logits, name="week6i_tinystories_tiny_gpt")


model = build_model()
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=LEARNING_RATE, weight_decay=0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

if LOAD_SAVED_WEIGHTS and os.path.exists(WEIGHTS_PATH):
    model.load_weights(WEIGHTS_PATH)

model.summary()

---
# Train

One epoch should begin producing recognizable story-like completions. Two epochs is the better default on a T4 if you can let it run for a few hours.

In [ ]:
#@title Train model { display-mode: "form" }
os.makedirs(SAVE_DIR, exist_ok=True)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        WEIGHTS_PATH,
        save_weights_only=True,
        save_best_only=True,
        monitor="val_loss",
        mode="min",
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

In [ ]:
#@title Plot training curve { display-mode: "form" }
plt.figure(figsize=(7, 3.5))
plt.plot(history.history["loss"], "o-", label="train")
plt.plot(history.history["val_loss"], "o--", label="validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("TinyStories Language Model Loss")
plt.legend()
plt.tight_layout()
plt.show()

---
# Try Text Completion

In [ ]:
#@title Generation utilities + prompt box { display-mode: "form" }
def prompt_tokens(prompt):
    ids = vectorizer([prompt]).numpy()[0]
    nonzero = np.where(ids > 0)[0]
    if len(nonzero) == 0:
        return []
    return list(ids[:nonzero[-1] + 1])


def generate_text(prompt, length=GENERATED_TOKENS, temperature=DEFAULT_TEMPERATURE):
    id_to_word = dict(enumerate(vocab))
    ids = prompt_tokens(prompt)
    if not ids:
        return "(type a prompt first)"

    prompt_len = len(ids)
    temperature = max(float(temperature), 1e-6)

    for _ in range(length):
        padded = np.zeros(SEQ_LEN, dtype="int32")
        context = ids[-SEQ_LEN:]
        padded[:len(context)] = context
        pred_pos = len(context) - 1

        logits = model.predict(padded[np.newaxis, :], verbose=0)[0][pred_pos]
        probs = tf.nn.softmax(logits / temperature).numpy().astype("float64")
        probs[0] = 0
        probs[1] = 0
        probs = probs / probs.sum()
        ids.append(int(np.random.choice(len(probs), p=probs)))

    prompt_words = [id_to_word.get(i, "") for i in ids[:prompt_len] if i > 1]
    generated_words = [id_to_word.get(i, "") for i in ids[prompt_len:] if i > 1]
    return f"<b>{' '.join(prompt_words)}</b> {' '.join(generated_words)}"


def make_prompt_widget():
    prompt_box = widgets.Text(
        value=DEFAULT_PROMPT,
        placeholder="Type a story beginning...",
        description="Prompt:",
        layout=widgets.Layout(width="560px"),
        style={"description_width": "70px"},
    )
    button = widgets.Button(description="Generate", button_style="primary")
    output = widgets.Output(layout=widgets.Layout(min_height="90px", padding="8px"))

    def run_generation(_=None):
        with output:
            output.clear_output(wait=True)
            display(HTML(f"<div style='font-size:15px; line-height:1.5'>{generate_text(prompt_box.value)}</div>"))

    button.on_click(run_generation)
    try:
        prompt_box.on_submit(run_generation)
    except Exception:
        pass

    display(widgets.HBox([prompt_box, button]))
    display(output)
    run_generation()


make_prompt_widget()

---
# Save Weights

Run this cell after training. It saves the model weights, vocabulary, config, and a zip file you can download from Colab.

In [ ]:
#@title Save weights and vocabulary { display-mode: "form" }
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_weights(WEIGHTS_PATH)

with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump(vocab, f)

config = {
    "dataset": "roneneldan/TinyStories",
    "train_stories": TRAIN_STORIES,
    "val_stories": VAL_STORIES,
    "vocab_size": VOCAB_SIZE,
    "seq_len": SEQ_LEN,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "generated_tokens": GENERATED_TOKENS,
    "default_temperature": DEFAULT_TEMPERATURE,
    "embed_dim": EMBED_DIM,
    "num_heads": NUM_HEADS,
    "ff_dim": FF_DIM,
    "num_blocks": NUM_BLOCKS,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _dirs, files in os.walk(SAVE_DIR):
        for filename in files:
            path = os.path.join(root, filename)
            zf.write(path, arcname=os.path.relpath(path, "."))

print(f"Saved weights and vocabulary to {ZIP_PATH}")

try:
    from google.colab import files
    files.download(ZIP_PATH)
except Exception:
    pass